# NB17 MEV and transaction ordering — Chapter 17

A chapter notebook for *Blockchain for Lawyers: A Reading Through Technology and Economics*. The code is the same as in the text.

- All code here is a **toy implementation** meant to make a principle visible. Do not use it with real assets, real private keys or real transactions.
- No external API is called, and everything runs without a network. Cells that need the bundled snapshot data carry that data inside the cell.
- Standard library only (run and checked on Python 3.13). There is no install cell.
- The outputs are the editor's own run, stored as produced. Running the cells again should give the same values (random seeds are fixed).

In [ ]:
# NB17-1: the sandwich — take Chapter 15's constant-product pool as it is and put in three transactions in order, "front, victim, back" (for study, in pool units)
import math

class ConstantProductPool:                                 # same class as NB15-1 in Chapter 15 (swap part only)
    def __init__(self, x, y, fee=0.003):
        self.x, self.y, self.fee = float(x), float(y), fee
    @property
    def price_x_in_y(self): return self.y / self.x
    def swap_x_for_y(self, dx):
        eff = dx * (1 - self.fee); dy = self.y * eff / (self.x + eff)
        self.x += dx; self.y -= dy; return dy
    def swap_y_for_x(self, dy):
        eff = dy * (1 - self.fee); dx = self.x * eff / (self.y + eff)
        self.y += dy; self.x -= dx; return dx

def sandwich(pool_x, pool_y, victim_y, attacker_y):
    """The victim buys X with Y victim_y. The attacker, in front of it, buys X with Y attacker_y and, behind it, sells all of that X back.
    Returns: (Y the attacker got back − Y put in, X the victim received, X the victim would have received alone)"""
    alone = ConstantProductPool(pool_x, pool_y).swap_y_for_x(victim_y)
    p = ConstantProductPool(pool_x, pool_y)
    front_x = p.swap_y_for_x(attacker_y) if attacker_y > 0 else 0.0     # ① front: the attacker buys
    victim_x = p.swap_y_for_x(victim_y)                                 # ② the victim buys, at the already raised price
    back_y = p.swap_x_for_y(front_x) if attacker_y > 0 else 0.0         # ③ back: the attacker sells
    return back_y - attacker_y, victim_x, alone

print("Pool X 1,000 / Y 1,000. The victim buys X with Y 100.")
print(f"{'attacker in Y':>14} {'attacker net profit (Y)':>24} {'victim got X':>14} {'X if alone':>12} {'victim loss X':>14} {'victim loss ratio':>18}")
for a in [0, 50, 100, 200, 400, 800, 2000, 5000, 20000, 100000]:
    profit, vx, alone = sandwich(1000, 1000, 100, a)
    print(f"{a:>14} {profit:>24.3f} {vx:>14.3f} {alone:>12.3f} {alone - vx:>14.3f} {1 - vx / alone:>18.2%}")

Pool X 1,000 / Y 1,000. The victim buys X with Y 100.
 attacker in Y  attacker net profit (Y)   victim got X   X if alone  victim loss X  victim loss ratio
             0                    0.000         90.661       90.661          0.000              0.00%
            50                    9.353         82.601       90.661          8.060              8.89%
           100                   17.368         75.570       90.661         15.091             16.65%
           200                   30.322         63.957       90.661         26.704             29.45%
           400                   48.047         47.526       90.661         43.135             47.58%
           800                   66.972         29.196       90.661         61.466             67.80%
          2000                   84.967         10.743       90.661         79.918             88.15%
          5000                   92.174          2.731       90.661         87.930             96.99%
         20000              

In [ ]:
# NB17-2: the victim's slippage tolerance caps the size of the attack — add a "minimum amount received" condition and find the attacker's best input (for study, in pool units)
def best_attack(pool_x, pool_y, victim_y, tolerance, fee=0.003, grid=2000):
    """Assume the victim attached to its transaction the condition "I must receive at least (1 − tolerance) of the X I would have received alone."
    Among the attacker inputs Y that do not break that condition (= the victim's transaction is not reverted), find by grid search the one with the largest net profit."""
    alone = ConstantProductPool(pool_x, pool_y, fee).swap_y_for_x(victim_y)
    min_out = alone * (1 - tolerance)
    best = (0.0, 0.0)                                          # (net profit, input)
    for i in range(1, grid + 1):
        a = pool_y * 2 * i / grid                              # sweep from 0 up to twice the pool's Y
        profit, vx, _ = sandwich(pool_x, pool_y, victim_y, a)
        if vx < min_out:                                       # if the victim's trade fails, the sandwich fails too (only the front trade remains, at the attacker's loss)
            break
        if profit > best[0]:
            best = (profit, a)
    return best, alone, min_out

print("Pool X 1,000 / Y 1,000. Victim order Y 100. The attacker's best input and net profit for each slippage tolerance the victim attached to the transaction:")
print(f"{'tolerance':>9} {'victim min X':>16} {'attacker best input Y':>21} {'attacker net profit (Y)':>24}")
for tol in [0.001, 0.005, 0.01, 0.02, 0.05, 0.10, 0.50]:
    (profit, a), alone, min_out = best_attack(1000, 1000, 100, tol)
    print(f"{tol:>9.1%} {min_out:>16.3f} {a:>21.1f} {profit:>24.3f}")
print("\nThe narrower the tolerance, the smaller the attack that can be slipped in (on this 1 Y grid it is 0 at 0.1%). Net profit is nearly proportional to the tolerance, and the best input grows faster than that.")

Pool X 1,000 / Y 1,000. Victim order Y 100. The attacker's best input and net profit for each slippage tolerance the victim attached to the transaction:
tolerance     victim min X attacker best input Y  attacker net profit (Y)
     0.1%           90.570                   0.0                    0.000
     0.5%           90.208                   2.0                    0.404
     1.0%           89.754                   5.0                    1.004
     2.0%           88.848                  10.0                    1.992
     5.0%           86.128                  27.0                    5.234
    10.0%           81.595                  56.0                   10.380
    50.0%           45.331                 434.0                   50.317

The narrower the tolerance, the smaller the attack that can be slipped in (on this 1 Y grid it is 0 at 0.1%). Net profit is nearly proportional to the tolerance, and the best input grows faster than that.
